# 🧠 第7周-Day1：数字员工总览与Agent行为设计

> **W7 数字员工架构深化周 · Day1**
>
> 数字员工不是聊天机器人，而是可执行、可协作的**数字劳动力**。
> 今天用 dataclass 模拟一个数字员工的完整行为配置。

**实验目标：**
1. 用 `dataclass` 构建数字员工行为配置（角色 + 权限 + 工具 + 护栏）
2. 可视化 System Prompt 四层结构
3. 对比不同配置下的「能力雷达图」

In [ ]:
# 配置 matplotlib 中文显示
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")


## 实验1：数字员工行为配置 dataclass

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Guardrail:
    name: str
    level: str
    description: str

@dataclass
class ToolConfig:
    name: str
    permission: str
    rate_limit: int

@dataclass
class DigitalEmployee:
    role: str
    rules: List[str] = field(default_factory=list)
    tools: List[ToolConfig] = field(default_factory=list)
    guardrails: List[Guardrail] = field(default_factory=list)
    output_format: str = "markdown"

    def summary(self):
        print(f"🔧 角色: {self.role}")
        print(f"📜 规则数: {len(self.rules)}")
        print(f"🛠️ 工具数: {len(self.tools)}")
        print(f"🛡️ 护栏数: {len(self.guardrails)}")
        for g in self.guardrails:
            print(f"  ⛔ [{g.level}] {g.name}: {g.description}")

analyst = DigitalEmployee(
    role="数据分析专员",
    rules=["只回答数据相关问题", "引用来源必须标注", "不编造数据"],
    tools=[ToolConfig("SQL查询","read",50), ToolConfig("图表生成","write",20), ToolConfig("邮件发送","write",5)],
    guardrails=[Guardrail("数据脱敏","hard","输出中不得包含手机号/身份证"), Guardrail("查询限制","soft","单次查询不超过1万行")]
)
analyst.summary()

## 实验2：System Prompt 四层结构可视化

In [ ]:
layers = ["角色定义\n(Identity)", "行为规则\n(Rules)", "安全护栏\n(Guardrails)", "输出格式\n(Format)"]
tokens = [120, 350, 280, 80]
colors = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(layers, tokens, color=colors, height=0.6)
ax.set_xlabel("Token 数（模拟）")
ax.set_title("System Prompt 四层结构")
for i, v in enumerate(tokens):
    ax.text(v + 5, i, str(v), va='center')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 实验3：能力雷达图对比不同数字员工

In [ ]:
from math import pi

categories = ["数据查询", "文本生成", "工具调用", "多步推理", "合规意识", "输出规范"]
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

emp_a = [9, 5, 7, 6, 8, 7] + [9]
emp_b = [4, 9, 5, 8, 6, 9] + [4]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles, emp_a, 'o-', linewidth=2, label="数据分析专员")
ax.fill(angles, emp_a, alpha=0.15)
ax.plot(angles, emp_b, 's-', linewidth=2, label="内容运营专员")
ax.fill(angles, emp_b, alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 10)
ax.set_title("数字员工能力雷达图")
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1))
plt.tight_layout()
plt.show()